In [2]:
from scipy.stats import norm
import numpy as np

In [3]:
#adapted from Octave's financial toolkit
def blsprice(Price, Strike, Rate, Time, Volatility):
    sigma_sqrtT = Volatility * np.sqrt (Time)

    d1 = 1 / sigma_sqrtT * (np.log(Price / Strike) + (Rate + Volatility**2 / 2) * Time)
    d2 = d1 - sigma_sqrtT

    phi1 = norm.cdf(d1)
    phi2 = norm.cdf(d2)
    disc = np.exp (-Rate * Time)
    F    = Price * np.exp ((Rate) * Time)

    Call = disc * (F * phi1 - Strike * phi2)
    Put  = disc * (Strike * (1 - phi2) + F * (phi1 - 1))
    return Call, Put

In [4]:
# Compute Black-Scholes option value using a binomial tree
# European OR American case
# uses vectorization for efficiency
def priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth):
    # S0 - current stock price
    # K - strike
    # T - expiry time
    # r - interest rate
    # sigma - volatility

    # opttype - 0 for a call, otherwise a put
    # countrytype - 0 for European, 1 for American
    # smooth - whether to smooth the payoff (1) or not (0)

    outersteps = 5;
    option_value_array = np.zeros(outersteps);

    for outer_count in np.arange(outersteps):

        Nsteps = 500*2**outer_count; # Nsteps - number of timesteps
        delt = T/Nsteps;

        # tree parameters
        u = np.exp(sigma * np.sqrt(delt) )
        d = 1/u;
        a = np.exp( r*delt )
        p = (a - d)/(u - d)

        #initialize the vector of final stock prices
        Sn = S0 * d**(np.arange(Nsteps,-1,-1)) * u**(np.arange(Nsteps+1))

        # determine payoff at t=T
        V = np.zeros(Nsteps+1) #just copy to set the size right

        if opttype == 0: #call
            V =  np.maximum( Sn - K, 0)
        else: #put
            V = np.maximum( K - Sn, 0)

        #vector operations - determine payoff values at N
        if(smooth == 0): #don't smooth
            if opttype == 0: #call
                V = np.maximum( Sn - K, 0)
            else: #put
                V = np.maximum( K - Sn, 0)

        else: # do smoothing
            srdt = sigma*np.sqrt(delt)
            if(opttype == 0): #call
                for i in np.arange(V.size):
                    if(Sn[i]*np.exp(srdt) < K):
                        V[i] = 0
                    elif(Sn[i]*np.exp(-srdt) > K):
                        V[i] = Sn[i]*(np.exp(srdt) - np.exp(-srdt))/(2*srdt)-K
                    else:
                        V[i] = 1/(2*srdt)*( Sn[i]*(np.exp(srdt)-K/Sn[i])  - K*(srdt - np.log(K/Sn[i])))
            else: #put
                for i in np.arange(V.size):
                    if(Sn[i]*np.exp(-srdt) > K):
                        V[i] = 0
                    elif(Sn[i]*np.exp(srdt) < K):
                        V[i] = K - Sn[i]*(np.exp(srdt) - np.exp(-srdt))/(2*srdt)
                    else:
                        V[i] = 1/(2*srdt)*(K * (np.log(K/Sn[i]) + srdt) - Sn[i]*(K/Sn[i] - np.exp(-srdt)))

        # backward recursion
        for n in np.arange(Nsteps, 0, -1):
            V = np.exp(-r*delt)*( p*V[1:n+1] + (1-p)*V[0:n] )
            if countrytype == 1: #American

                #Direct computation of S
                #Sn = S0 * d**(np.arange(n-1,-1,-1)) * u**(np.arange(n))

                #Backtracking from the final data - same result
                Sn = Sn / u 
                Sn = Sn[1:]

                #Alternatively, Justin's notes suggest precomputing a vector of the final S 
                #values at the start and using clever indexing.

                if opttype == 0: #call
                    payoff = np.maximum( Sn - K, 0)
                elif opttype == 1: #put
                    payoff = np.maximum( K - Sn, 0)

                V = np.maximum(V, payoff)

        value = V[0]
        option_value_array[outer_count] = value

    print("dt\t\tValue\t\t\tChange\t\t\tRatio")
    for outer_count in np.arange(outersteps):
        Nsteps = 500*2**outer_count; # Nsteps - number of timesteps
        delt = T/Nsteps;
        if outer_count == 0:
            print(str(delt) + "\t\t" + str(option_value_array[outer_count]))
        elif outer_count == 1:
            dif = option_value_array[outer_count] - option_value_array[outer_count-1]
            print(str(delt) + "\t\t" + str(option_value_array[outer_count]) + "\t" + str(dif))
        else:
            dif = option_value_array[outer_count] - option_value_array[outer_count-1]
            dif2 = option_value_array[outer_count-1] - option_value_array[outer_count-2]
            ratio = dif2/dif
            print(str(delt) + "\t" + str(option_value_array[outer_count]) + "\t" + str(dif) + "\t" + str(ratio))

    [call, put] = blsprice(S0, K, r, T, sigma)        

    if countrytype == 0:
        if opttype == 0:
            print("Exact\t\t" + str(call) )
        else:
            print("Exact\t\t" + str(put) )



# Question 1

In [5]:
#Table 1 data
S0 = 100 # S0 - current stock price
K = 100 # K - strike
T = 1 # T - expiry time
r = .04 # r - interest rate
sigma = .32 # sigma - volatility

In [6]:
#European call, no smoothing 

opttype = 0 # opttype - 0 for a call, otherwise a put
countrytype = 0 #countrytype - 0 for European, 1 for American
smooth = 0 #whether to smooth the payoff or not

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)


dt		Value			Change			Ratio
0.002		14.513319919417471
0.001		14.516457291797295	0.0031373723798235886
0.0005	14.518026273384256	0.0015689815869617263	1.999623453771049
0.00025	14.518810837859961	0.000784564475704741	1.9998121703794665
0.000125	14.51920313853851	0.00039230067854845174	1.9999059869274278
Exact		14.519595451443333


In [7]:
#European put, no smoothing 

opttype = 1 # opttype - 0 for a call, otherwise a put
countrytype = 0 #countrytype - 0 for European, 1 for American
smooth = 0 #whether to smooth the payoff or not

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)


dt		Value			Change			Ratio
0.002		10.592263834646982
0.001		10.595401207019856	0.0031373723728744807
0.0005	10.596970188631337	0.0015689816114807797	1.9996234180931405
0.00025	10.597754753131191	0.0007845644998543122	1.999812140075326
0.000125	10.5981470537607	0.0003923006295085685	1.9999062984862634
Exact		10.598539366675652


In [8]:
#American put, no smoothing 

opttype = 1 # opttype - 0 for a call, otherwise a put
countrytype = 1 #countrytype - 0 for European, 1 for American
smooth = 0 #whether to smooth the payoff or not

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)

dt		Value			Change			Ratio
0.002		10.991823139809135
0.001		10.993458100264535	0.001634960455399792
0.0005	10.994280111872063	0.0008220116075285944	1.9889749006286612
0.00025	10.994684282443904	0.0004041705718407229	2.033823501263065
0.000125	10.99488666612025	0.00020238367634561882	1.9970512401924374


Based on the observed data, we see the ratio is consistently around 2, which implies that the method is first order accurate. This is indeed consistent with equation (1) and our derivation 
in the question.

# Question 2

In [9]:
S0 = 91 # the starting stock price no longer matches the strike

In [10]:
#American put, *no* smoothing 

opttype = 1 # opttype - 0 for a call, otherwise a put
countrytype = 1 #countrytype - 0 for European, 1 for American
smooth = 0 #whether to smooth the payoff or not

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)

dt		Value			Change			Ratio
0.002		15.243289869687164
0.001		15.241372201552448	-0.0019176681347161662
0.0005	15.240496184120055	-0.0008760174323931835	2.189075312665077
0.00025	15.23981370024388	-0.0006824838761740892	1.2835723494363218
0.000125	15.239405541883368	-0.00040815836051244503	1.6721055898921855


We observe that without smoothing and when S0 doesn't match K, we don't see proper convergence. The observed error ratio is very erratic!

In [11]:
#American put, *with* smoothing 

opttype = 1 # opttype - 0 for a call, otherwise a put
countrytype = 1 #countrytype - 0 for European, 1 for American
smooth = 1 #whether to smooth the payoff or not

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)

dt		Value			Change			Ratio
0.002		15.244212333236108
0.001		15.241795692902864	-0.002416640333244402
0.0005	15.24056038537269	-0.001235307530173202	1.9563066477102797
0.00025	15.239916540380042	-0.000643844992648468	1.9186412013421774
0.000125	15.239597666686514	-0.00031887369352823214	2.019122322461084


When we enable smoothing, we see that we once again achieve the expected apparent convergence at first order.

In [12]:
opttype = 1 # opttype - 0 for a call, otherwise a put
countrytype = 1 #countrytype - 0 for European, 1 for American
smooth = 1 #whether to smooth the payoff or not
S0 = 92
K = 100
T = 1
sigma = 0.3
r = 0.02

priceoption(S0, K, T, r, sigma, opttype, countrytype, smooth)

dt		Value			Change			Ratio
0.002		14.859875016496346
0.001		14.85734399323617	-0.0025310232601754024
0.0005	14.855967410082618	-0.0013765831535526019	1.8386272225135778
0.00025	14.855298099456798	-0.0006693106258204296	2.056717913099332
0.000125	14.854959415383998	-0.00033868407279946666	1.9762093336367943
